In [11]:
# 导入树叶数据集
import torch
import torchvision
from torch.utils.data import DataLoader,random_split
from torchvision import transforms
from tqdm import tqdm
import os
import csv


In [12]:
def load_data_classify_leaves(batch_size,resize=None):
    trans = [
        transforms.RandomHorizontalFlip(),  # 随机水平翻转
        transforms.RandomVerticalFlip(),  # 随机垂直翻转
        transforms.RandomRotation(30),  # 随机旋转，最大角度为30°
        transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 随机亮度和对比度调整
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),  # 随机平移和缩放
        transforms.ToTensor(),  # 转换为Tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 归一化
    ]
    if resize:
        trans.insert(0,transforms.Resize(resize))
    trans = transforms.Compose(trans)
    root_dir = "../datasets/classify-leaves/train" # 总长度为18353
    dataset = torchvision.datasets.ImageFolder(root=root_dir, transform=trans)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    return (DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=4,pin_memory=True,prefetch_factor=4,persistent_workers=True),
            DataLoader(val_dataset, batch_size=batch_size, shuffle=False,num_workers=4,pin_memory=True,prefetch_factor=4)
    )
# 计算时间
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_iter, val_iter = load_data_classify_leaves(128, resize=None)
# for i,(X,y) in enumerate(train_iter): # 这里费时间
#     X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
#     print(i,X.shape,y.shape,X.dtype,y.dtype)
#     break

In [13]:
import torch
from torch import nn

class Residual(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, X):
        Y = self.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return self.relu(Y)
# blk = Residual(3, 6,use_1x1conv=True,strides=2)

# X = torch.rand(1, 3, 6, 6)
# blk(X).shape

def resnet_block(in_channels, out_channels, num_residuals,
                 first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(in_channels, out_channels, use_1x1conv=True,
                                strides=2))
        else:
            blk.append(Residual(out_channels, out_channels))
    return blk


layer1 = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64),nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)
# 一个残差块有两个卷积，一个层有两个残差快。所以一个层有四个卷积，综述4*4 = 16个
layer2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
layer3 = nn.Sequential(*resnet_block(64, 128, 2))
layer4 = nn.Sequential(*resnet_block(128, 256, 2))
layer5 = nn.Sequential(*resnet_block(256, 512, 2))
net = nn.Sequential(layer1,layer2, layer3, layer4, layer5, 
                    nn.AdaptiveAvgPool2d((1,1)),nn.Flatten(),nn.Linear(512,176))

X = torch.rand(1,3,224,224)
# Y  =  net(X)
# Y.shape

for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 128, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 512, 1, 1])
Flatten output shape:	 torch.Size([1, 512])
Linear output shape:	 torch.Size([1, 176])


In [14]:
def train_fromKK(net, train_iter, test_iter, num_epochs, lr, device):
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.kaiming_normal_(m.weight)
    net.apply(init_weights)
    print('training on', device)
    net.to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        net.train()
        train_loss_sum, train_acc_sum,num_samples = 0,0,0
        with tqdm(train_iter, desc=f"Epoch {epoch+1}/{num_epochs}") as pbar:  
            for X, y in pbar:
                optimizer.zero_grad()
                X,y = X.to(device),y.to(device)
                y_hat = net(X)
                l = loss(y_hat, y)
                l.backward()
                optimizer.step()
                train_loss_sum += l.item() * X.shape[0]
                train_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
                num_samples += X.shape[0]
                pbar.set_postfix(loss=l.item(), acc=train_acc_sum / num_samples)
        train_loss = train_loss_sum / num_samples
        train_acc = train_acc_sum / num_samples
        if (epoch+1) % 3 == 0:
            net.eval()  # 评估模式
            val_acc_sum, val_samples = 0, 0
            with torch.no_grad():
                for X, y in val_iter: # 这里也很费时间
                    X, y = X.to(device), y.to(device)
                    y_hat = net(X)
                    val_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
                    val_samples += X.shape[0]
            val_acc = val_acc_sum / val_samples
            print(f"______ | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        else: 
            print(f"______ | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        
     

In [15]:
lr,num_epochs =0.001,1
train_fromKK(net,train_iter,val_iter,num_epochs,lr,device)


training on cuda


Epoch 1/1: 100%|██████████| 115/115 [00:46<00:00,  2.46it/s, acc=0.0571, loss=3.72]

______ | Train Loss: 4.6262 | Train Acc: 0.0571


In [22]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = [os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith(('.jpg', '.png', '.jpeg'))]
        # print(self.image_paths)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = torchvision.datasets.folder.pil_loader(img_path)  # 加载图片
        if self.transform:
            image = self.transform(image)
        return image, img_path  # 返回图片和文件路径

# 加载无标签图片的DataLoader
def load_test_data(batch_size, image_dir, resize=None):
    trans = [
        transforms.RandomHorizontalFlip(),  # 随机水平翻转
        transforms.RandomVerticalFlip(),  # 随机垂直翻转
        transforms.RandomRotation(30),  # 随机旋转，最大角度为30°
        transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 随机亮度和对比度调整
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),  # 随机平移和缩放
        transforms.ToTensor(),  # 转换为Tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 归一化
    ]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)

    dataset = CustomDataset(image_dir=image_dir, transform=trans) # 替换了ImageFolder
    print(len(dataset))
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    return test_loader

image_dir = "../datasets/classify-leaves/test"  # 测试图片文件夹路径
test_iter = load_test_data(batch_size=8, image_dir=image_dir, resize=None)
for X, img_path in test_iter:
    print(img_path)
    break

8800
('../datasets/classify-leaves/test\\18353.jpg', '../datasets/classify-leaves/test\\18354.jpg', '../datasets/classify-leaves/test\\18355.jpg', '../datasets/classify-leaves/test\\18356.jpg', '../datasets/classify-leaves/test\\18357.jpg', '../datasets/classify-leaves/test\\18358.jpg', '../datasets/classify-leaves/test\\18359.jpg', '../datasets/classify-leaves/test\\18360.jpg')


In [33]:

# 生成submission.csv
def generate_submission(net, test_iter, device, filename='submission.csv'):
    net.eval()  # 设置模型为评估模式
    predictions = []
    labels = os.listdir("../datasets/classify-leaves/train")

    with torch.no_grad():
        for X, img_path in test_iter:  # 遍历测试集
            X = X.to(device)
            y_hat = net(X)  # 获取预测结果
            predicted_labels = y_hat.argmax(dim=1).cpu().numpy()  # 获取每个样本的预测标签           
            predicted_labels= [labels[i] for i in predicted_labels]
            for i in range(X.shape[0]):
                file_name = img_path[i].split('/')[-1]  # 获取图片文件名
                file_name = "images/"+file_name.split('\\')[-1]  
                predictions.append([file_name, predicted_labels[i]])  # 保存文件名与预测标签
            

    # 将结果保存到 CSV 文件中
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image', 'label'])  # 写入表头
        writer.writerows(predictions)  # 写入预测结果

    print(f"Submission file '{filename}' has been saved.")



# 调用函数生成预测结果并保存
generate_submission(net, test_iter, device='cuda')  # 假设你在 GPU 上训练

['magnolia_virginiana', 'prunus_sargentii', 'albizia_julibrissin', 'pinus_flexilis', 'oxydendrum_arboreum', 'albizia_julibrissin', 'magnolia_virginiana', 'albizia_julibrissin']
['crataegus_viridis', 'maclura_pomifera', 'quercus_cerris', 'magnolia_virginiana', 'magnolia_virginiana', 'magnolia_virginiana', 'salix_babylonica', 'metasequoia_glyptostroboides']
['cryptomeria_japonica', 'magnolia_virginiana', 'juniperus_virginiana', 'pinus_flexilis', 'robinia_pseudo-acacia', 'magnolia_virginiana', 'ailanthus_altissima', 'ptelea_trifoliata']
['broussonettia_papyrifera', 'magnolia_virginiana', 'maclura_pomifera', 'pinus_flexilis', 'quercus_montana', 'diospyros_virginiana', 'albizia_julibrissin', 'oxydendrum_arboreum']
['robinia_pseudo-acacia', 'acer_palmatum', 'prunus_virginiana', 'magnolia_virginiana', 'acer_palmatum', 'acer_palmatum', 'picea_pungens', 'picea_pungens']
['salix_babylonica', 'magnolia_virginiana', 'ulmus_rubra', 'pinus_flexilis', 'taxodium_distichum', 'ulmus_rubra', 'quercus_cer